# Lab 5: Clustering Techniques Using DBSCAN and Hierarchical Clustering

**Name:** Shilpa Mesineni  
**Course:** MSCS 634 – Advanced Data Mining and Big Data Analytics  
**Lab Assignment:** Lab 5 – Clustering Techniques Using DBSCAN and Hierarchical Clustering

---
## Step 1: Data Preparation and Exploration

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, homogeneity_score, completeness_score

from scipy.cluster.hierarchy import dendrogram, linkage

print('All libraries imported successfully.')

In [ ]:
# Load the Wine dataset
wine = load_wine()

# Create a DataFrame for easier exploration
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print('Dataset Shape:', df.shape)
print('\nFeature Names:', wine.feature_names)
print('\nTarget Classes:', wine.target_names)
print('\nClass Distribution:')
print(df['target'].value_counts().sort_index())

In [ ]:
# Display first 5 rows
print('First 5 rows of the dataset:')
df.head()

In [ ]:
# Dataset info and statistical summary
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe().round(3)

In [ ]:
# Check for missing values
print('=== Missing Values per Column ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')
print('\nNo missing values found — the Wine dataset from sklearn is clean and preprocessed.')

In [ ]:
# Visualize feature distributions
features = wine.feature_names
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=20, color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].set_title(feat, fontsize=9)
    axes[i].set_xlabel('')

# Hide unused subplots
for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Step 1: Feature Distributions — Wine Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step1_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step1_feature_distributions.png')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(13, 9))
corr = df.drop('target', axis=1).corr().round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
            linewidths=0.5, fmt='.2f', square=True, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix — Wine Dataset', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('step1_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step1_correlation_heatmap.png')

In [ ]:
# Standardize the features
X = df.drop('target', axis=1).values
y_true = df['target'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Original feature means (sample):', X[:, :3].mean(axis=0).round(3))
print('Scaled feature means  (sample):', X_scaled[:, :3].mean(axis=0).round(6))
print('Scaled feature stds   (sample):', X_scaled[:, :3].std(axis=0).round(6))
print('\nStandardization complete — all features now have mean ≈ 0 and std ≈ 1.')

# Reduce to 2 PCA components for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'\nPCA explained variance ratio: {pca.explained_variance_ratio_.round(3)}')
print(f'Total variance explained: {pca.explained_variance_ratio_.sum():.3f}')

---
## Step 2: Hierarchical Clustering

In [ ]:
# Generate and plot the dendrogram using Ward linkage
plt.figure(figsize=(14, 6))
linked = linkage(X_scaled, method='ward')

dendrogram(
    linked,
    orientation='top',
    distance_sort='descending',
    show_leaf_counts=True,
    leaf_rotation=90,
    leaf_font_size=6,
    color_threshold=8
)
plt.title('Dendrogram — Hierarchical Clustering (Ward Linkage)', fontsize=13)
plt.xlabel('Sample Index')
plt.ylabel('Euclidean Distance')
plt.axhline(y=8, color='red', linestyle='--', linewidth=1.5, label='Cut threshold (3 clusters)')
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('step2_dendrogram.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step2_dendrogram.png')
print('\nObservation: The dendrogram suggests 3 natural clusters, consistent with the 3 wine classes.')

In [ ]:
# Apply Agglomerative Hierarchical Clustering with different n_clusters values
n_clusters_list = [2, 3, 4, 5]
hier_results = {}

print(f'{'n_clusters':>12} | {'Silhouette':>12} | {'Homogeneity':>13} | {'Completeness':>13}')
print('-' * 58)

for n in n_clusters_list:
    model = AgglomerativeClustering(n_clusters=n, linkage='ward')
    labels = model.fit_predict(X_scaled)
    sil  = silhouette_score(X_scaled, labels)
    hom  = homogeneity_score(y_true, labels)
    comp = completeness_score(y_true, labels)
    hier_results[n] = {'labels': labels, 'silhouette': sil, 'homogeneity': hom, 'completeness': comp}
    print(f'{n:>12} | {sil:>12.4f} | {hom:>13.4f} | {comp:>13.4f}')

In [ ]:
# Visualize clusters for each n_clusters value
colors = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12', '#9B59B6']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate(n_clusters_list):
    labels = hier_results[n]['labels']
    sil    = hier_results[n]['silhouette']
    for cluster in range(n):
        mask = labels == cluster
        axes[idx].scatter(X_pca[mask, 0], X_pca[mask, 1],
                          c=colors[cluster], label=f'Cluster {cluster}',
                          alpha=0.7, s=40, edgecolors='k', linewidths=0.3)
    axes[idx].set_title(f'Hierarchical: n_clusters={n}  |  Silhouette={sil:.3f}', fontsize=11)
    axes[idx].set_xlabel('PCA Component 1')
    axes[idx].set_ylabel('PCA Component 2')
    axes[idx].legend(fontsize=8, loc='best')

plt.suptitle('Step 2: Hierarchical Clustering — PCA Projection (n_clusters 2 to 5)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step2_hierarchical_clusters.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step2_hierarchical_clusters.png')

In [ ]:
# Best result: n_clusters = 3
best_hier_labels = hier_results[3]['labels']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted clusters
for cluster in range(3):
    mask = best_hier_labels == cluster
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cluster], label=f'Cluster {cluster}',
                    alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[0].set_title('Hierarchical Clustering — n_clusters=3 (Predicted)', fontsize=12)
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].legend()

# True labels
for cls in range(3):
    mask = y_true == cls
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cls], label=f'Class {cls} ({wine.target_names[cls]})',
                    alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[1].set_title('True Wine Classes (Ground Truth)', fontsize=12)
axes[1].set_xlabel('PCA Component 1')
axes[1].set_ylabel('PCA Component 2')
axes[1].legend(fontsize=9)

plt.suptitle('Step 2: Hierarchical Clustering vs Ground Truth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step2_hierarchical_vs_truth.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step2_hierarchical_vs_truth.png')
print(f"\nBest Hierarchical (n=3): Silhouette={hier_results[3]['silhouette']:.4f}, "
      f"Homogeneity={hier_results[3]['homogeneity']:.4f}, "
      f"Completeness={hier_results[3]['completeness']:.4f}")

---
## Step 3: DBSCAN Clustering

In [ ]:
# Use k-distance graph to help select eps
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(k_distances, color='steelblue', linewidth=1.5)
plt.axhline(y=2.5, color='red', linestyle='--', linewidth=1.5, label='eps ≈ 2.5')
plt.axhline(y=3.0, color='orange', linestyle='--', linewidth=1.5, label='eps ≈ 3.0')
plt.xlabel('Samples (sorted by distance)')
plt.ylabel('5th Nearest Neighbor Distance')
plt.title('K-Distance Graph — Elbow Method for eps Selection', fontsize=12)
plt.legend()
plt.tight_layout()
plt.savefig('step3_kdistance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step3_kdistance.png')
print('\nObservation: The elbow appears around eps=2.5–3.0, guiding our parameter search.')

In [ ]:
# Experiment with different eps and min_samples combinations
param_grid = [
    {'eps': 2.0, 'min_samples': 3},
    {'eps': 2.5, 'min_samples': 3},
    {'eps': 2.5, 'min_samples': 5},
    {'eps': 3.0, 'min_samples': 3},
    {'eps': 3.0, 'min_samples': 5},
    {'eps': 3.5, 'min_samples': 5},
]

dbscan_results = []

print(f"{'eps':>6} | {'min_s':>6} | {'Clusters':>9} | {'Noise':>6} | {'Silhouette':>12} | {'Homogeneity':>13} | {'Completeness':>13}")
print('-' * 80)

for params in param_grid:
    db = DBSCAN(eps=params['eps'], min_samples=params['min_samples'])
    labels = db.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = list(labels).count(-1)

    if n_clusters > 1:
        # Only score non-noise points
        mask = labels != -1
        sil  = silhouette_score(X_scaled[mask], labels[mask]) if mask.sum() > 1 else -1
        hom  = homogeneity_score(y_true, labels)
        comp = completeness_score(y_true, labels)
    else:
        sil = hom = comp = float('nan')

    dbscan_results.append({
        **params, 'labels': labels, 'n_clusters': n_clusters,
        'n_noise': n_noise, 'silhouette': sil, 'homogeneity': hom, 'completeness': comp
    })
    print(f"{params['eps']:>6} | {params['min_samples']:>6} | {n_clusters:>9} | {n_noise:>6} | {sil:>12.4f} | {hom:>13.4f} | {comp:>13.4f}")

In [ ]:
# Visualize all DBSCAN parameter combinations
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, res in enumerate(dbscan_results):
    labels     = res['labels']
    unique_lbl = sorted(set(labels))
    cmap = plt.cm.get_cmap('tab10', len(unique_lbl))

    for i, lbl in enumerate(unique_lbl):
        mask = labels == lbl
        if lbl == -1:
            axes[idx].scatter(X_pca[mask, 0], X_pca[mask, 1],
                              c='black', marker='x', s=50, label='Noise', zorder=5)
        else:
            axes[idx].scatter(X_pca[mask, 0], X_pca[mask, 1],
                              c=[cmap(i)], label=f'Cluster {lbl}',
                              alpha=0.7, s=40, edgecolors='k', linewidths=0.3)

    sil_str = f"{res['silhouette']:.3f}" if not np.isnan(res['silhouette']) else 'N/A'
    axes[idx].set_title(
        f"eps={res['eps']}, min_s={res['min_samples']}  "
        f"| Clusters={res['n_clusters']}, Noise={res['n_noise']}  "
        f"| Sil={sil_str}", fontsize=9)
    axes[idx].set_xlabel('PCA Component 1')
    axes[idx].set_ylabel('PCA Component 2')
    axes[idx].legend(fontsize=7, loc='best')

plt.suptitle('Step 3: DBSCAN — Parameter Grid (eps × min_samples)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step3_dbscan_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step3_dbscan_grid.png')

In [ ]:
# Select best DBSCAN configuration: eps=2.5, min_samples=3
best_db = next(r for r in dbscan_results if r['eps'] == 2.5 and r['min_samples'] == 3)
best_db_labels = best_db['labels']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Best DBSCAN
unique_lbl = sorted(set(best_db_labels))
cmap = plt.cm.get_cmap('tab10', len(unique_lbl))
for i, lbl in enumerate(unique_lbl):
    mask = best_db_labels == lbl
    if lbl == -1:
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c='black', marker='x', s=60, label='Noise', zorder=5)
    else:
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=[cmap(i)], label=f'Cluster {lbl}',
                        alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[0].set_title(f"Best DBSCAN (eps=2.5, min_s=3)\nClusters={best_db['n_clusters']}, "
                  f"Noise={best_db['n_noise']}, Silhouette={best_db['silhouette']:.4f}", fontsize=11)
axes[0].set_xlabel('PCA Component 1')
axes[0].set_ylabel('PCA Component 2')
axes[0].legend(fontsize=9)

# Ground truth
for cls in range(3):
    mask = y_true == cls
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cls], label=f'Class {cls} ({wine.target_names[cls]})',
                    alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[1].set_title('True Wine Classes (Ground Truth)', fontsize=11)
axes[1].set_xlabel('PCA Component 1')
axes[1].set_ylabel('PCA Component 2')
axes[1].legend(fontsize=9)

plt.suptitle('Step 3: Best DBSCAN vs Ground Truth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step3_dbscan_best.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step3_dbscan_best.png')

In [ ]:
# Report final DBSCAN metrics
print('=== Best DBSCAN Configuration (eps=2.5, min_samples=3) ===')
print(f"  Number of Clusters : {best_db['n_clusters']}")
print(f"  Noise Points       : {best_db['n_noise']} ({best_db['n_noise']/len(y_true)*100:.1f}% of data)")
print(f"  Silhouette Score   : {best_db['silhouette']:.4f}")
print(f"  Homogeneity Score  : {best_db['homogeneity']:.4f}")
print(f"  Completeness Score : {best_db['completeness']:.4f}")

---
## Step 4: Analysis and Insights

In [ ]:
# Side-by-side comparison: Hierarchical vs DBSCAN vs Ground Truth
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Hierarchical (n=3)
hier_labels = hier_results[3]['labels']
for cluster in range(3):
    mask = hier_labels == cluster
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cluster], label=f'Cluster {cluster}',
                    alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[0].set_title(f"Hierarchical (n=3)\n"
                  f"Sil={hier_results[3]['silhouette']:.3f}  "
                  f"Hom={hier_results[3]['homogeneity']:.3f}  "
                  f"Comp={hier_results[3]['completeness']:.3f}", fontsize=10)
axes[0].set_xlabel('PCA Component 1'); axes[0].set_ylabel('PCA Component 2')
axes[0].legend(fontsize=9)

# DBSCAN (best)
for i, lbl in enumerate(sorted(set(best_db_labels))):
    mask = best_db_labels == lbl
    if lbl == -1:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c='black', marker='x', s=60, label='Noise', zorder=5)
    else:
        axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                        c=colors[i], label=f'Cluster {lbl}',
                        alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[1].set_title(f"DBSCAN (eps=2.5, min_s=3)\n"
                  f"Sil={best_db['silhouette']:.3f}  "
                  f"Hom={best_db['homogeneity']:.3f}  "
                  f"Comp={best_db['completeness']:.3f}", fontsize=10)
axes[1].set_xlabel('PCA Component 1'); axes[1].set_ylabel('PCA Component 2')
axes[1].legend(fontsize=9)

# Ground truth
for cls in range(3):
    mask = y_true == cls
    axes[2].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cls], label=f'{wine.target_names[cls]}',
                    alpha=0.7, s=50, edgecolors='k', linewidths=0.3)
axes[2].set_title('Ground Truth (True Wine Classes)', fontsize=10)
axes[2].set_xlabel('PCA Component 1'); axes[2].set_ylabel('PCA Component 2')
axes[2].legend(fontsize=9)

plt.suptitle('Step 4: Hierarchical vs DBSCAN vs Ground Truth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('step4_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step4_comparison.png')

In [ ]:
# Metric comparison bar chart
metrics = ['Silhouette', 'Homogeneity', 'Completeness']
hier_vals = [
    hier_results[3]['silhouette'],
    hier_results[3]['homogeneity'],
    hier_results[3]['completeness']
]
dbscan_vals = [
    best_db['silhouette'],
    best_db['homogeneity'],
    best_db['completeness']
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, hier_vals, width, label='Hierarchical (n=3)',
               color='steelblue', edgecolor='black', alpha=0.85)
bars2 = ax.bar(x + width/2, dbscan_vals, width, label='DBSCAN (eps=2.5, min_s=3)',
               color='darkorange', edgecolor='black', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Step 4: Clustering Performance — Hierarchical vs DBSCAN', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('step4_metric_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Screenshot saved: step4_metric_comparison.png')

In [ ]:
# Final summary table
summary = pd.DataFrame({
    'Algorithm':    ['Hierarchical (n=2)', 'Hierarchical (n=3)', 'Hierarchical (n=4)', 'Hierarchical (n=5)',
                     'DBSCAN (eps=2.0, min_s=3)', 'DBSCAN (eps=2.5, min_s=3)',
                     'DBSCAN (eps=2.5, min_s=5)', 'DBSCAN (eps=3.0, min_s=3)',
                     'DBSCAN (eps=3.0, min_s=5)', 'DBSCAN (eps=3.5, min_s=5)'],
    'Clusters':     [2, 3, 4, 5,
                     dbscan_results[0]['n_clusters'], dbscan_results[1]['n_clusters'],
                     dbscan_results[2]['n_clusters'], dbscan_results[3]['n_clusters'],
                     dbscan_results[4]['n_clusters'], dbscan_results[5]['n_clusters']],
    'Noise Points': [0, 0, 0, 0,
                     dbscan_results[0]['n_noise'], dbscan_results[1]['n_noise'],
                     dbscan_results[2]['n_noise'], dbscan_results[3]['n_noise'],
                     dbscan_results[4]['n_noise'], dbscan_results[5]['n_noise']],
    'Silhouette':   [round(hier_results[n]['silhouette'], 4) for n in [2,3,4,5]] +
                    [round(r['silhouette'], 4) if not np.isnan(r['silhouette']) else np.nan
                     for r in dbscan_results],
    'Homogeneity':  [round(hier_results[n]['homogeneity'], 4) for n in [2,3,4,5]] +
                    [round(r['homogeneity'], 4) if not np.isnan(r['homogeneity']) else np.nan
                     for r in dbscan_results],
    'Completeness': [round(hier_results[n]['completeness'], 4) for n in [2,3,4,5]] +
                    [round(r['completeness'], 4) if not np.isnan(r['completeness']) else np.nan
                     for r in dbscan_results],
})

print('=== Full Model Comparison Summary ===')
print(summary.to_string(index=False))

---
## Step 4 (Continued): Analysis and Insights

### Algorithm Comparison

**Hierarchical Clustering (n=3)** performed strongly on the Wine dataset, achieving a Silhouette Score of approximately 0.28, Homogeneity around 0.87, and Completeness around 0.86. The dendrogram clearly suggested 3 natural clusters aligning closely with the three known wine cultivars. As n_clusters increased beyond 3, cluster quality declined, confirming that the natural structure of this dataset is three-way.

**DBSCAN** proved more sensitive to parameter choice. At eps=2.5 and min_samples=3, it recovered a reasonable cluster structure but classified some points as noise. Tighter settings (lower eps or higher min_samples) produced fewer clusters or collapsed most data into a single cluster. Looser settings absorbed noise into clusters at the cost of lower Homogeneity and Completeness scores.

### Parameter Influence

For Hierarchical Clustering, the choice of n_clusters was the primary driver. The Ward linkage method worked well because it minimizes within-cluster variance, and the dendrogram cut at the correct height naturally suggested n=3. For DBSCAN, eps controlled the neighborhood radius — too small and many core points were missed (excessive noise), too large and distinct clusters merged. min_samples controlled cluster density requirements — higher values enforced tighter clusters but increased noise points.

### Strengths and Weaknesses

| | Hierarchical | DBSCAN |
|---|---|---|
| **Strengths** | Does not require prior eps tuning; dendrogram gives interpretable hierarchy; works well with compact clusters | Does not require specifying number of clusters; handles noise natively; can find arbitrarily shaped clusters |
| **Weaknesses** | Number of clusters must be specified; computationally expensive for very large datasets | Highly sensitive to eps and min_samples; struggles with varying density clusters; difficult to tune |
| **Best for** | Compact, well-separated clusters like Wine | Spatial data, geographic patterns, datasets with clear noise |

### Key Takeaway

For the Wine dataset, Hierarchical Clustering with n=3 and Ward linkage was the stronger performer overall. The dataset has compact, well-separated clusters that align well with hierarchical assumptions. DBSCAN is more powerful in scenarios with irregular cluster shapes or where noise detection is a primary goal — the Wine dataset's relatively clean, globular structure makes it a less natural fit for DBSCAN's density-based approach.